# OSort / NWB Data Extraction — DA057 Session

This notebook covers **all three data tiers** from the ephys_data folder:

| Tier | Path | Contents | Curation |
|------|------|----------|----------|
| **0** | `ephys_data/sort/A00_5-16_2_cells_final copy.xlsx` | Human curation log — per-channel quality scores + accepted cluster IDs | Gold standard |
| **1a** | `ephys_data/sort/5.1/A_ss{N}_sorted_new.mat` | Raw OSort output at 5.1× std threshold (all 32 channels) | None |
| **1b** | `ephys_data/sort/5.2/A_ss{N}_sorted_new.mat` | Raw OSort output at 5.2× std threshold | None |
| **2** | `ephys_data/sort/final/A_ss{N}_{Min|Max}_sorted_new.mat` | Hand-curated OSort output (5 files, channels with real neurons) | ✓ Human |

**OSort cluster ID conventions:**
- `0` / `1` → unclustered / first events (edge cases, ignore)
- `1NNN–3NNN` → OSort template cluster IDs (assigned sequentially)
- `99999999` → OSort hard-coded noise sentinel (oscillation / saturations / cross-talk)
- `useNegative` → accepted as isolated single units (SUs)
- `useNegativeExcluded` *(final/ only)* → rejected by human after visual inspection

**Waveform encoding:**  raw ADU × `scalingFactor` × 1e6 → **µV**  
**Timestamps:** UNIX microseconds → `(t - t0) / 1e6` → **relative seconds**


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────── #
import sys, subprocess
for pkg in ['scipy', 'openpyxl']:
    try: __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import scipy.io as sio
import numpy as np
import pandas as pd
from pathlib import Path
import datetime, warnings
warnings.filterwarnings('ignore')

SORT_DIR  = Path('ephys_data/sort')
XLSX_FILE = SORT_DIR / 'A00_5-16_2_cells_final copy.xlsx'
DIR_51    = SORT_DIR / '5.1'
DIR_52    = SORT_DIR / '5.2'
DIR_FINAL = SORT_DIR / 'final'

# Scaling factor is stored per-file in 5.1/5.2; apply to convert ADU → µV
SCALING_V_PER_ADU  = 9.155273e-8    # from A_ss1_sorted_new.mat (hardware constant)
SCALING_UV_PER_ADU = SCALING_V_PER_ADU * 1e6

# OSort noise sentinel
NOISE_ID = 99999999

print(f"SORT_DIR  : {SORT_DIR.resolve()}")
print(f"5.1 files : {len(list(DIR_51.glob('*.mat')))}")
print(f"5.2 files : {len(list(DIR_52.glob('*.mat')))}")
print(f"final files: {len(list(DIR_FINAL.glob('*.mat')))}")
print(f"xlsx found : {XLSX_FILE.exists()}")


SORT_DIR  : /Users/marco/Cursor_Folder/Cursor_Codespace/spike_discrim/ephys_data/sort
5.1 files : 32
5.2 files : 32
final files: 5
xlsx found : True


## Tier 0 — Human Curation Log (XLSX → DataFrame + NPZ)

`A00_5-16_2_cells_final copy.xlsx` is the experimenter's per-channel scorecard.  
One row per electrode channel (32 total). Key columns:

| Column | Meaning |
|--------|---------|
| `File` | Channel number (1–32) → maps to `A_ss{File}_sorted_new.mat` |
| `HS` | Signal quality code: 0=no signal, 2=ephys no neurons, 4=has neurons |
| `Nr SUA` | Number of isolated single units on this channel |
| `Th used` | OSort threshold that yielded the best separation (5.1 or 5.2) |
| `Clusters to Use` | Specific OSort cluster IDs accepted as SUs |
| `Max`/`Min` | Were positive (Max) or negative (Min) deflections sorted? |


In [3]:
# ── Parse XLSX curation log ───────────────────────────────────────────────── #
# header is at row index 9; actual column mapping (confirmed by inspection):
#   File   → channel number 1–32
#   HS     → headstage letter (hardware, not quality)
#   Nr SUA → number of isolated single units (NaN or int)
#   Th used → OSort threshold used (NaN / '5.1' / '5.2' / '5.1;5.2')
#   Clusters to Use → OSort cluster IDs (NaN or comma/semicolon-delimited string)
#   Notes  → *** quality code: 0=no signal, 2=ephys/no neurons, 4=has neurons ***

raw = pd.read_excel(XLSX_FILE, sheet_name='1', header=9)
raw.columns = [str(c).strip() for c in raw.columns]

# Keep only the 32 channel rows (File = 1–32, integer)
curation_df = (raw[raw['File'].apply(lambda x: str(x).strip().isdigit())]
               .copy()
               .reset_index(drop=True))

curation_df['channel']         = curation_df['File'].astype(int)
curation_df['quality_code']    = pd.to_numeric(curation_df['Notes'],    errors='coerce').fillna(0).astype(int)
curation_df['n_sua']           = pd.to_numeric(curation_df['Nr SUA'],   errors='coerce').fillna(0).astype(int)
curation_df['threshold_used']  = curation_df['Th used'].astype(str).str.strip()
curation_df['cluster_ids_raw'] = curation_df['Clusters to Use'].astype(str).str.strip()
curation_df['has_neurons']     = curation_df['quality_code'] == 4

# Parse cluster IDs to lists of ints (handles '2557, 2907, 2911+2939' notation —
# the +ID suffix means a merged cluster; we keep only the primary ID before '+')
def parse_clusters(s):
    if s in ('nan', '', 'NaN', 'None'): return []
    ids = []
    for tok in s.replace(';', ',').split(','):
        primary = tok.split('+')[0].strip()
        if primary.isdigit():
            ids.append(int(primary))
    return ids

curation_df['cluster_ids'] = curation_df['cluster_ids_raw'].apply(parse_clusters)

print("Quality code legend:  0=no signal  2=ephys/no neurons  4=has neurons")
print()
active = curation_df[curation_df['has_neurons']]
print(f"Channels with confirmed neurons: {len(active)}")
print()
print(active[['channel', 'quality_code', 'n_sua', 'threshold_used',
              'cluster_ids_raw', 'cluster_ids']].to_string(index=False))


Quality code legend:  0=no signal  2=ephys/no neurons  4=has neurons

Channels with confirmed neurons: 4

 channel  quality_code  n_sua threshold_used                        cluster_ids_raw                    cluster_ids
       8             4      2            5.1                             1112, 1137                   [1112, 1137]
      11             4      1            5.2                                   2498                         [2498]
      12             4      2        5.1;5.2                              3303;2962                   [3303, 2962]
      13             4      5            5.1 2557, 2907, 2911+2939, 2961, 2963+2978 [2557, 2907, 2911, 2961, 2963]


## Tier 1 — Raw OSort Output (5.1 / 5.2 folders)

Each `.mat` file is one electrode channel at one threshold pass.  
**These are uncurated** — any cluster could be a real neuron or noise artifact.

### MAT file key reference

| Key | Shape | dtype | Units | Description |
|-----|-------|-------|-------|-------------|
| `newSpikesNegative` | (N, 256) | float64 | ADU | Raw 256-sample waveform snippets (negative deflection) |
| `allSpikesCorrFree` | (N, 256) | float64 | ADU | Artifact-corrected version of the same waveforms |
| `newTimestampsNegative` | (1, N) | float64 | UNIX µs | Timestamp of each event in microseconds since epoch |
| `assignedNegative` | (1, N) | int32 | — | OSort cluster ID per event |
| `useNegative` | (K, 1) | uint16 | — | Cluster IDs OSort marked as single units |
| `noiseTraces` | (84, 1) | float64 | ADU | 84-sample noise reference template |
| `scalingFactor` | (1,1) | float64 | V/ADU | ADC scaling: multiply × 1e6 for µV |
| `stdEstimateOrig` | (1,1) | float64 | ADU | Pre-filtering noise std (×scalingFactor×1e6 → µV) |
| `paramsUsed` | (1,2) | float64 | — | [0, threshold_multiplier] (5.1 or 5.2) |
| `newSpikesPositive` | (0,0) | uint8 | — | Empty — positive polarity spikes (not used here) |

### Cluster ID semantics
```
0          → unclustered  (OSort edge-case, 1 event, ignore)
1–98999998 → OSort template IDs (sequential integer counter)
useNegative values → accepted SUs (by OSort automated criterion only)
99999999   → noise sentinel (OSort hard-rejected: oscillations, saturations, cross-talk)
```


In [4]:
def load_osort_mat(path):
    """Load an OSort .mat file and return a clean dict with decoded arrays.
    
    Returns
    -------
    dict with keys:
        waveforms_adu   (N,256) float64 — raw ADU waveforms
        waveforms_uv    (N,256) float64 — µV waveforms (× scalingFactor × 1e6)
        waveforms_corr  (N,256) float64 — artifact-corrected ADU waveforms
        timestamps_us   (N,)   float64 — UNIX timestamps in microseconds
        timestamps_rel  (N,)   float64 — seconds from first event
        cluster_ids     (N,)   int32   — OSort cluster ID per event
        su_ids          list[int]       — IDs OSort accepted as single units
        excluded_ids    list[int]       — IDs human rejected (final/ only)
        noise_template  (84,)  float64 — noise reference waveform (ADU)
        scaling         float           — V/ADU conversion factor
        std_uv          float           — estimated noise std (µV)
        n_total         int
        n_su            int             — events in accepted SU clusters
        n_noise         int             — events with ID == 99999999
        n_excluded      int             — events in manually excluded clusters
    """
    mat   = sio.loadmat(str(path), squeeze_me=False)
    
    w_adu = mat['newSpikesNegative']                        # (N,256)
    w_cor = mat['allSpikesCorrFree']                        # (N,256)
    t_us  = mat['newTimestampsNegative'].flatten()          # (N,)
    ids   = mat['assignedNegative'].flatten().astype(int)   # (N,)
    su    = mat['useNegative'].flatten().astype(int).tolist()
    excl  = mat.get('useNegativeExcluded',
                    np.zeros((0,1), dtype=np.uint16)).flatten().astype(int).tolist()
    noise_tmpl = mat['noiseTraces'].flatten()               # (84,)
    scaling    = float(mat.get('scalingFactor',
                               np.array([[SCALING_V_PER_ADU]]))[0, 0])
    std_adu    = float(mat['stdEstimateOrig'][0, 0])
    
    t0         = t_us[0]
    w_uv       = w_adu * scaling * 1e6
    
    su_mask    = np.isin(ids, su)
    excl_mask  = np.isin(ids, excl)
    noise_mask = ids == NOISE_ID
    
    return {
        'waveforms_adu':  w_adu,
        'waveforms_uv':   w_uv,
        'waveforms_corr': w_cor,
        'timestamps_us':  t_us,
        'timestamps_rel': (t_us - t0) / 1e6,
        'cluster_ids':    ids,
        'su_ids':         su,
        'excluded_ids':   excl,
        'noise_template': noise_tmpl,
        'scaling':        scaling,
        'std_uv':         std_adu * scaling * 1e6,
        'n_total':        len(ids),
        'n_su':           int(su_mask.sum()),
        'n_noise':        int(noise_mask.sum()),
        'n_excluded':     int(excl_mask.sum()),
    }


# ── Demo: inspect ss1 from 5.1 ─────────────────────────────────────────────  #
d = load_osort_mat(DIR_51 / 'A_ss1_sorted_new.mat')

print("── A_ss1_sorted_new.mat (5.1, uncurated) ──────────────────────────────")
print(f"  Total events          : {d['n_total']}")
print(f"  Accepted SU clusters  : {d['su_ids']}")
print(f"  OSort noise (99999999): {d['n_noise']}")
print(f"  Recording span        : {d['timestamps_rel'][-1]:.1f} s")
print(f"  Noise σ estimate      : {d['std_uv']:.2f} µV")
print(f"  Scaling factor        : {d['scaling']:.4e} V/ADU")
print()

# Mean waveform per accepted cluster
print("── Mean waveforms of accepted SUs (µV) ────────────────────────────────")
for cid in d['su_ids']:
    mask   = d['cluster_ids'] == cid
    mean_w = d['waveforms_uv'][mask].mean(axis=0)
    std_w  = d['waveforms_uv'][mask].std(axis=0)
    trough = mean_w.argmin()
    print(f"  cluster {cid:5d}: n={mask.sum():4d}  "
          f"trough={mean_w[trough]:7.1f} ± {std_w[trough]:.1f} µV  "
          f"@ sample {trough}/256")

print()
# Noise events
noise_mask = d['cluster_ids'] == NOISE_ID
print(f"── Noise events (99999999): n={d['n_noise']} ──────────────────────────")
print(f"   Amplitude range: {d['waveforms_uv'][noise_mask].min():.1f} – "
      f"{d['waveforms_uv'][noise_mask].max():.1f} µV  "
      f"(note: these are real physiological noise for training!)")


── A_ss1_sorted_new.mat (5.1, uncurated) ──────────────────────────────
  Total events          : 3025
  Accepted SU clusters  : [1441, 1643, 1653]
  OSort noise (99999999): 1949
  Recording span        : 1944.6 s
  Noise σ estimate      : 3.64 µV
  Scaling factor        : 9.1553e-08 V/ADU

── Mean waveforms of accepted SUs (µV) ────────────────────────────────
  cluster  1441: n= 113  trough=  -23.1 ± 4.8 µV  @ sample 121/256
  cluster  1643: n= 412  trough=  -10.1 ± 4.5 µV  @ sample 65/256
  cluster  1653: n= 550  trough=  -12.3 ± 4.0 µV  @ sample 125/256

── Noise events (99999999): n=1949 ──────────────────────────
   Amplitude range: -1152.2 – 777.9 µV  (note: these are real physiological noise for training!)


## Tier 2 — Hand-Curated Files (final/)

These 5 files are the **ground truth** for this recording session.  
Only channels that had visually verified isolated neurons were retained:

| File | Channel | SU cluster(s) | Threshold | n_SU | n_excluded |
|------|---------|---------------|-----------|------|------------|
| A_ss8_Max | ss8 | 1112, 1137 | 5.1 | 3,721 | ~6,695 |
| A_ss11_Min | ss11 | 2498 | 5.2 | 982 | ~14,434 |
| A_ss12_Max | ss12 | 3303 | 5.1 | 2,557 | ~14,160 |
| A_ss12_Min | ss12 | 2962 | 5.2 | 3,493 | ~13,597 |
| A_ss13_Max | ss13 | 2557, 2907, 2911, 2961, 2963 | 5.1 | 27,392 | ~3,542 |

`useNegativeExcluded` = cluster IDs the human explicitly rejected.  
These are **real sub-threshold events that look spike-like** — exactly what you want as hard negatives for training the discriminator input layer.


In [5]:
# ── Inspect all final/ files ──────────────────────────────────────────────── #
final_files = sorted(DIR_FINAL.glob('*.mat'))

rows = []
for path in final_files:
    d   = load_osort_mat(path)
    tag = path.stem   # e.g. A_ss11_Min_sorted_new

    print(f"\n{'─'*62}")
    print(f"  {path.name}")
    print(f"  Total events: {d['n_total']:6d}  |  "
          f"Accepted SUs: {d['n_su']:5d}  |  "
          f"Excluded: {d['n_excluded']:5d}  |  "
          f"Noise(99M): {d['n_noise']:5d}")
    print(f"  SU cluster IDs:       {d['su_ids']}")
    print(f"  Excluded cluster IDs: {d['excluded_ids']}")
    print(f"  Recording span: {d['timestamps_rel'][-1]:.1f} s  |  "
          f"Noise σ: {d['std_uv']:.2f} µV")

    # Per-cluster breakdown
    for cid in sorted(np.unique(d['cluster_ids'])):
        mask  = d['cluster_ids'] == cid
        label = ('SU'       if cid in d['su_ids'] else
                 'EXCL'     if cid in d['excluded_ids'] else
                 'NOISE'    if cid == NOISE_ID else 'OTHER')
        mean_w = d['waveforms_uv'][mask].mean(axis=0)
        rows.append({
            'file':        path.stem,
            'cluster_id':  int(cid),
            'label':       label,
            'n_events':    int(mask.sum()),
            'trough_uv':   float(mean_w.min()),
            'trough_samp': int(mean_w.argmin()),
            'peak_uv':     float(mean_w.max()),
        })
        print(f"    [{label:<5}] cluster {cid:>8d}: n={mask.sum():5d}  "
              f"trough={mean_w.min():7.1f} µV @ samp {mean_w.argmin()}")

cluster_summary = pd.DataFrame(rows)
print(f"\n\nTotal labelled events across all final files:")
print(cluster_summary.groupby('label')['n_events'].sum().to_string())



──────────────────────────────────────────────────────────────
  A_ss11_Min_sorted_new.mat
  Total events:  15417  |  Accepted SUs:   982  |  Excluded: 12953  |  Noise(99M):  1481
  SU cluster IDs:       [2498]
  Excluded cluster IDs: [2412, 2748, 2817, 2837, 2869, 2924, 2930, 2933, 2949, 2970]
  Recording span: 1950.2 s  |  Noise σ: 4.88 µV
    [OTHER] cluster        0: n=    1  trough=  -24.0 µV @ samp 94
    [EXCL ] cluster     2412: n=   96  trough=  -53.4 µV @ samp 94
    [SU   ] cluster     2498: n=  982  trough=  -32.9 µV @ samp 94
    [EXCL ] cluster     2748: n=  194  trough=  -16.4 µV @ samp 94
    [EXCL ] cluster     2817: n=  231  trough=  -33.4 µV @ samp 94
    [EXCL ] cluster     2837: n=  109  trough=  -16.6 µV @ samp 94
    [EXCL ] cluster     2869: n=  392  trough=  -29.9 µV @ samp 94
    [EXCL ] cluster     2924: n=  330  trough=  -22.0 µV @ samp 94
    [EXCL ] cluster     2930: n=  696  trough=  -21.1 µV @ samp 94
    [EXCL ] cluster     2933: n= 1573  trough=  -21.

## Build Pipeline-Compatible Dataset from Curated Data

The spike_discrim pipeline expects `data/synthetic/waveforms.npz` with:
```
waveforms    (N, n_samples) float32 — voltage snippets
class_labels (N,)           int32   — 1=spike, 0=noise
unit_ids     (N,)           int32   — per-event unit identity (0=noise)
```

Strategy from the curated files:
- **Class 1 (spikes)**: `useNegative` events — human-verified isolated SUs  
- **Class 0 (noise)**: `useNegativeExcluded` + `99999999` events — real physiological noise  
- **Sub-sample** to 64 samples centred on the trough (pipeline default) from the 256-sample OSort window


In [6]:
def extract_window(waveforms_uv, n_out=64, align='trough'):
    """Sub-sample 256-sample OSort snippets to n_out samples centred on the trough.
    
    OSort stores 256 samples; the pipeline uses 64.  The trough sits at varying
    positions (median ~sample 94 in ss11, ~121 in ss1, etc.) so we align by the
    trough of each snippet's mean, then extract a fixed window.
    
    Parameters
    ----------
    waveforms_uv : (N,256) float64
    n_out        : output snippet length (default 64 to match pipeline)
    align        : 'trough' (per-snippet), 'mean_trough' (align to ensemble mean)
    
    Returns
    -------
    (N, n_out) float32
    """
    N, T = waveforms_uv.shape
    pre  = n_out // 4           # samples before trough  (16 of 64)
    out  = np.zeros((N, n_out), dtype=np.float32)

    if align == 'mean_trough':
        centre = int(np.argmin(waveforms_uv.mean(axis=0)))
        start  = max(0, centre - pre)
        end    = start + n_out
        if end > T:
            start = T - n_out
            end   = T
        out[:] = waveforms_uv[:, start:end].astype(np.float32)
    else:   # per-snippet trough alignment
        for i in range(N):
            c  = int(np.argmin(waveforms_uv[i]))
            s  = max(0, c - pre)
            e  = s + n_out
            if e > T:
                s = T - n_out
                e = T
            out[i] = waveforms_uv[i, s:e].astype(np.float32)

    return out


# ── Build labelled dataset from all final/ files ───────────────────────────── #
all_waves, all_labels, all_unit_ids, all_meta = [], [], [], []
unit_counter = 1    # unit_id ≥ 1 for SUs; 0 = noise

for path in sorted(DIR_FINAL.glob('*.mat')):
    d      = load_osort_mat(path)
    ids    = d['cluster_ids']
    w_uv   = d['waveforms_uv']
    t_rel  = d['timestamps_rel']
    
    # ── Spikes: accepted SU clusters ─────────────────────────────────────── #
    for cid in d['su_ids']:
        mask = ids == cid
        w    = extract_window(w_uv[mask], n_out=64, align='mean_trough')
        all_waves.append(w)
        all_labels.append(np.ones(w.shape[0], dtype=np.int32))
        all_unit_ids.append(np.full(w.shape[0], unit_counter, dtype=np.int32))
        for i in np.where(mask)[0]:
            all_meta.append({
                'source_file': path.name, 'cluster_id': cid,
                'label': 'SU', 'unit_id': unit_counter,
                'timestamp_s': float(t_rel[i]),
            })
        print(f"  [SU  ] {path.name}  cluster {cid}  → unit_id={unit_counter}  n={mask.sum()}")
        unit_counter += 1
    
    # ── Noise: excluded clusters + OSort noise sentinel ───────────────────── #
    noise_mask = np.isin(ids, d['excluded_ids']) | (ids == NOISE_ID)
    if noise_mask.sum() > 0:
        w = extract_window(w_uv[noise_mask], n_out=64, align='mean_trough')
        all_waves.append(w)
        all_labels.append(np.zeros(w.shape[0], dtype=np.int32))
        all_unit_ids.append(np.zeros(w.shape[0], dtype=np.int32))
        for i in np.where(noise_mask)[0]:
            cid_i = ids[i]
            src   = 'excluded' if cid_i in d['excluded_ids'] else 'noise_sentinel'
            all_meta.append({
                'source_file': path.name, 'cluster_id': int(cid_i),
                'label': 'NOISE', 'unit_id': 0,
                'timestamp_s': float(t_rel[i]),
            })
        print(f"  [NOISE] {path.name}  excluded+sentinel  n={noise_mask.sum()}")

waveforms  = np.concatenate(all_waves,     axis=0)
labels     = np.concatenate(all_labels,    axis=0)
unit_ids   = np.concatenate(all_unit_ids,  axis=0)
meta_df    = pd.DataFrame(all_meta)

# Shuffle
rng = np.random.default_rng(42)
idx = rng.permutation(len(waveforms))
waveforms  = waveforms[idx]
labels     = labels[idx]
unit_ids   = unit_ids[idx]
meta_df    = meta_df.iloc[idx].reset_index(drop=True)

print(f"\n── Dataset summary ─────────────────────────────────────────────────")
print(f"   Total snippets : {len(waveforms):,}")
print(f"   Spikes (1)     : {labels.sum():,}")
print(f"   Noise  (0)     : {(labels==0).sum():,}")
print(f"   Unique SU IDs  : {np.unique(unit_ids[unit_ids>0])}")
print(f"   Snippet shape  : {waveforms.shape}  float32")


  [SU  ] A_ss11_Min_sorted_new.mat  cluster 2498  → unit_id=1  n=982
  [NOISE] A_ss11_Min_sorted_new.mat  excluded+sentinel  n=14434
  [SU  ] A_ss12_Max_sorted_new.mat  cluster 3303  → unit_id=2  n=2557
  [NOISE] A_ss12_Max_sorted_new.mat  excluded+sentinel  n=16258
  [SU  ] A_ss12_Min_sorted_new.mat  cluster 2962  → unit_id=3  n=3493
  [NOISE] A_ss12_Min_sorted_new.mat  excluded+sentinel  n=14542
  [SU  ] A_ss13_Max_sorted_new.mat  cluster 2557  → unit_id=4  n=707
  [SU  ] A_ss13_Max_sorted_new.mat  cluster 2907  → unit_id=5  n=5914
  [SU  ] A_ss13_Max_sorted_new.mat  cluster 2911  → unit_id=6  n=3539
  [SU  ] A_ss13_Max_sorted_new.mat  cluster 2961  → unit_id=7  n=2175
  [SU  ] A_ss13_Max_sorted_new.mat  cluster 2963  → unit_id=8  n=15057
  [NOISE] A_ss13_Max_sorted_new.mat  excluded+sentinel  n=5478
  [SU  ] A_ss8_Max_sorted_new.mat  cluster 1112  → unit_id=9  n=742
  [SU  ] A_ss8_Max_sorted_new.mat  cluster 1137  → unit_id=10  n=2979
  [NOISE] A_ss8_Max_sorted_new.mat  excluded+sen

In [ ]:
import os

out_dir = Path('data/real_units')
out_dir.mkdir(parents=True, exist_ok=True)

# Primary archive: pipeline-compatible
np.savez_compressed(
    out_dir / 'waveforms_real.npz',
    waveforms    = waveforms,     # (N, 64) float32  µV
    class_labels = labels,        # (N,)    int32    0=noise  1=spike
    unit_ids     = unit_ids,      # (N,)    int32    0=noise  1..K=SU
)

# Metadata sidecar (CSV for human readability)
meta_df.to_csv(out_dir / 'waveforms_real_meta.csv', index=False)

# Curation log (for cross-referencing with xlsx)
np.savez_compressed(
    'ephys_data/sort/curation_log.npz',
    channels         = curation_df['channel'].values.astype(np.int32),
    quality_code     = curation_df['quality_code'].values.astype(np.int32),
    n_sua            = curation_df['n_sua'].values.astype(np.int32),
    has_neurons      = (curation_df['quality_code'] == 4).values,
    threshold_used   = curation_df['threshold_used'].values,       # object (strings)
    cluster_ids_raw  = curation_df['cluster_ids_raw'].values,      # object (strings)
)

sz_kb = os.path.getsize(out_dir / 'waveforms_real.npz') / 1024
print(f"Saved  data/real_units/waveforms_real.npz   {sz_kb:.1f} KB")
print(f"Saved  data/real_units/waveforms_real_meta.csv")
print(f"Saved  ephys_data/sort/curation_log.npz")
print()

# Quick sanity check: load back and verify
loaded = np.load(outx_dir / 'waveforms_real.npz')
print("── Load-back check ─────────────────────────────────────────────────")
print(f"   waveforms     : {loaded['waveforms'].shape}  {loaded['waveforms'].dtype}")
print(f"   class_labels  : {loaded['class_labels'].shape}  unique={np.unique(loaded['class_labels'])}")
print(f"   unit_ids      : {loaded['unit_ids'].shape}   unique={np.unique(loaded['unit_ids'])}")
print(f"   spike balance : {loaded['class_labels'].mean()*100:.1f}% spikes")


Saved  data/synthetic/waveforms_real.npz   22028.0 KB
Saved  data/synthetic/waveforms_real_meta.csv
Saved  ephys_data/sort/curation_log.npz

── Load-back check ─────────────────────────────────────────────────
   waveforms     : (96006, 64)  float32
   class_labels  : (96006,)  unique=[0 1]
   unit_ids      : (96006,)   unique=[ 0  1  2  3  4  5  6  7  8  9 10]
   spike balance : 39.7% spikes
